# Seoul Restaurant Commercial-Area Survival Analysis

서울시 일반음식점 인허가 데이터와 서울시 상권분석서비스 데이터를 결합하여 음식점의 생존기간과 폐업 위험 요인을 분석합니다.

## Analysis pipeline
1. Load and preprocess restaurant licensing data
2. Define survival duration and closure event
3. Filter major restaurant categories
4. Spatially join restaurants to commercial areas
5. Merge store, sales, floating-population, and resident-population features
6. Estimate Kaplan–Meier survival curves
7. Fit Cox proportional hazards models
8. Visualize hazard ratios with a forest plot

> **Data note:** Raw Seoul Open Data files are not embedded in this notebook. Update the paths in the configuration cells to match your environment.

## 0. Environment Setup

필요한 패키지를 설치하고 공통 라이브러리를 불러옵니다.

In [ ]:
import glob
import os
import unicodedata

import chardet
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import multivariate_logrank_test


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FIGURE_DIR = PROJECT_ROOT / "figures"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", DATA_DIR)


In [ ]:
# Optional: inspect file encoding
import chardet

file_path = DATA_DIR / "서울시 일반음식점 인허가 정보.csv"

with open(file_path, "rb") as f:
    raw = f.read(200000)   # 앞부분 20만 바이트만 확인

result = chardet.detect(raw)

print(result)

In [ ]:
import pandas as pd

file_path = DATA_DIR / "서울시 일반음식점 인허가 정보.csv"

df = pd.read_csv(
    file_path,
    encoding="euc-kr",
    encoding_errors="replace",
    low_memory=False
)

print(df.head())
print(df.shape)

In [ ]:
print("행 개수:", df.shape[0])
print("열 개수:", df.shape[1])

for i, col in enumerate(df.columns):
    print(i, col)

## 1. Restaurant Data Preprocessing

분석기간을 2021-01-01부터 2025-12-31까지로 설정하고, 관찰 종료일 이전 폐업 여부와 생존기간을 생성합니다.

In [ ]:
# 1. 필요한 컬럼 선택
cols = [
    "관리번호",
    "인허가일자",
    "영업상태명",
    "상세영업상태명",
    "폐업일자",
    "소재지면적",
    "지번주소",
    "도로명주소",
    "사업장명",
    "업태구분명",
    "좌표정보(X)",
    "좌표정보(Y)"
]

food = df[cols].copy()


# 2. 날짜 변환
food["인허가일자"] = pd.to_datetime(
    food["인허가일자"],
    errors="coerce"
)

food["폐업일자"] = pd.to_datetime(
    food["폐업일자"],
    errors="coerce"
)


# 3. 날짜 결측 확인
print("인허가일자 결측:", food["인허가일자"].isna().sum())
print("폐업일자 결측:", food["폐업일자"].isna().sum())


# 4. 영업 상태 확인
print("\n[영업상태명]")
print(food["영업상태명"].value_counts(dropna=False))

print("\n[상세영업상태명]")
print(food["상세영업상태명"].value_counts(dropna=False))

In [ ]:
# 5. 분석 기간 설정
start_date = pd.Timestamp("2021-01-01")
end_date   = pd.Timestamp("2025-12-31")

food_2021 = food[
    (food["인허가일자"] >= start_date) &
    (food["인허가일자"] <= end_date)
].copy()

print("2021~2025 개업 음식점 수:", len(food_2021))

In [ ]:
# 관찰 종료일
observation_end = pd.Timestamp("2025-12-31")


# 2025-12-31 이전 또는 당일에 폐업한 경우에만 event = 1
food_2021["event"] = (
    food_2021["폐업일자"].notna()
    & (food_2021["폐업일자"] <= observation_end)
).astype(int)


print(food_2021["event"].value_counts())

In [ ]:
food_2021["종료일"] = observation_end

food_2021.loc[
    food_2021["event"] == 1,
    "종료일"
] = food_2021.loc[
    food_2021["event"] == 1,
    "폐업일자"
]

In [ ]:
food_2021["duration_days"] = (
    food_2021["종료일"]
    - food_2021["인허가일자"]
).dt.days

In [ ]:
print("전체:", len(food_2021))

print("\n[event]")
print(food_2021["event"].value_counts())

print(
    "\n생존기간 <= 0:",
    (food_2021["duration_days"] <= 0).sum()
)

print(
    "\n생존기간 결측:",
    food_2021["duration_days"].isna().sum()
)

print("\n[event별 생존기간]")
print(
    food_2021.groupby("event")["duration_days"].describe()
)

In [ ]:
print(food_2021["event"].value_counts())

print(food_2021.groupby("event")["duration_days"].describe())

print("2025년 이후 폐업:", closed_after_2025.sum())

### 1.1 Clean survival records

생존기간이 0 이하인 레코드를 제거합니다.

In [ ]:
food_clean = food_2021[
    food_2021["duration_days"] > 0
].copy()

print("정제 전:", len(food_2021))
print("정제 후:", len(food_clean))
print("제거된 행:", len(food_2021) - len(food_clean))

In [ ]:
print(food_clean["event"].value_counts())

print(
    food_clean.groupby("event")["duration_days"].describe()
)

In [ ]:
print(food_clean["업태구분명"].value_counts(dropna=False).head(30))

### 1.2 Select restaurant categories

분석 대상 업태를 한식, 중국식, 일식, 경양식, 분식, 호프/통닭으로 제한합니다.

In [ ]:
major_types = [
    "한식",
    "중국식",
    "일식",
    "경양식",
    "분식",
    "호프/통닭"
]

food_clean = food_clean[
    food_clean["업태구분명"].isin(major_types)
].copy()

print(food_clean["업태구분명"].value_counts())
print("남은 음식점 수:", len(food_clean))

### 1.3 Clean coordinates

공간조인을 위해 좌표형을 숫자로 변환하고 좌표 결측치를 제거합니다.

In [ ]:
# 8. 좌표 결측 확인

print("전체 음식점 수:", len(food_clean))

print(
    "X좌표 결측:",
    food_clean["좌표정보(X)"].isna().sum()
)

print(
    "Y좌표 결측:",
    food_clean["좌표정보(Y)"].isna().sum()
)

In [ ]:
print(food_clean[["좌표정보(X)", "좌표정보(Y)"]].describe())

In [ ]:
print(
    food_clean[
        [
            "사업장명",
            "도로명주소",
            "좌표정보(X)",
            "좌표정보(Y)"
        ]
    ].head(20)
)

In [ ]:
print(food_clean["좌표정보(X)"].dtype)
print(food_clean["좌표정보(Y)"].dtype)

In [ ]:
food_clean["좌표정보(X)"] = pd.to_numeric(
    food_clean["좌표정보(X)"],
    errors="coerce"
)

food_clean["좌표정보(Y)"] = pd.to_numeric(
    food_clean["좌표정보(Y)"],
    errors="coerce"
)

In [ ]:
print("X좌표 결측:", food_clean["좌표정보(X)"].isna().sum())
print("Y좌표 결측:", food_clean["좌표정보(Y)"].isna().sum())

In [ ]:
food_geo = food_clean.dropna(
    subset=["좌표정보(X)", "좌표정보(Y)"]
).copy()

print("좌표 정제 전:", len(food_clean))
print("좌표 정제 후:", len(food_geo))
print("제거된 행:", len(food_clean) - len(food_geo))

## 2. Spatial Join with Seoul Commercial Areas

음식점 좌표를 GeoDataFrame으로 변환한 뒤 상권 폴리곤과 공간조인합니다. 중첩 상권은 우선순위 규칙으로 하나만 선택하고 관광특구 여부를 별도 변수로 보존합니다.

In [ ]:
import geopandas as gpd

restaurant_gdf = gpd.GeoDataFrame(
    food_geo,
    geometry=gpd.points_from_xy(
        food_geo["좌표정보(X)"],
        food_geo["좌표정보(Y)"]
    ),
    crs="EPSG:5174"
)

print(restaurant_gdf.head())
print("음식점 CRS:", restaurant_gdf.crs)

In [ ]:
import glob
import os

# 각 확장자 파일 찾기
extensions = ["shp", "shx", "dbf", "prj", "cpg"]

for ext in extensions:
    files = glob.glob(str(DATA_DIR / f"*.{ext}"))
    print(ext, files)

In [ ]:
import glob
import os

extensions = ["shp", "shx", "dbf", "prj", "cpg"]

for ext in extensions:
    files = glob.glob(str(DATA_DIR / f"*.{ext}"))

    if len(files) > 0:
        old = files[0]
        new = DATA_DIR / f"commercial_area.{ext}"

        os.rename(old, new)
        print(old, "->", new)

In [ ]:
print(glob.glob(str(DATA_DIR / "commercial_area.*")))

In [ ]:
import geopandas as gpd

shp_path = DATA_DIR / "commercial_area.shp"

commercial_gdf = gpd.read_file(shp_path)

print(commercial_gdf.head())
print("상권 데이터 크기:", commercial_gdf.shape)
print("상권 CRS:", commercial_gdf.crs)
print("컬럼:", commercial_gdf.columns.tolist())

In [ ]:
print(commercial_gdf.shape)
print(commercial_gdf.crs)
print(commercial_gdf.columns.tolist())

In [ ]:
restaurant_gdf = restaurant_gdf.to_crs(
    commercial_gdf.crs
)

print("음식점 CRS:", restaurant_gdf.crs)
print("상권 CRS:", commercial_gdf.crs)

In [ ]:
commercial_small = commercial_gdf[
    [
        "TRDAR_CD",
        "TRDAR_CD_N",
        "TRDAR_SE_1",
        "SIGNGU_CD",
        "SIGNGU_CD_",
        "ADSTRD_CD",
        "ADSTRD_CD_",
        "geometry"
    ]
].copy()

In [ ]:
joined = gpd.sjoin(
    restaurant_gdf,
    commercial_small,
    how="left",
    predicate="within"
)

In [ ]:
print("조인 전:", len(restaurant_gdf))
print("조인 후:", len(joined))

print("상권 매칭 성공:", joined["TRDAR_CD"].notna().sum())
print("상권 매칭 실패:", joined["TRDAR_CD"].isna().sum())

print(
    "매칭률:",
    round(joined["TRDAR_CD"].notna().mean() * 100, 2),
    "%"
)

In [ ]:
print(
    joined[
        [
            "사업장명",
            "도로명주소",
            "업태구분명",
            "TRDAR_CD",
            "TRDAR_CD_N"
        ]
    ].head(20)
)

In [ ]:
print("관리번호 중복:", joined["관리번호"].duplicated().sum())

In [ ]:
print(commercial_gdf["TRDAR_SE_1"].value_counts())

In [ ]:
print(joined["TRDAR_SE_1"].value_counts(dropna=False))

In [ ]:
dup_mask = joined["관리번호"].duplicated(keep=False)

duplicates = joined[dup_mask].copy()

print("중복된 행 수:", len(duplicates))
print("중복된 음식점 수:", duplicates["관리번호"].nunique())

In [ ]:
print(
    duplicates[
        [
            "관리번호",
            "사업장명",
            "도로명주소",
            "TRDAR_SE_1",
            "TRDAR_CD",
            "TRDAR_CD_N"
        ]
    ]
    .sort_values("관리번호")
    .head(50)
)

In [ ]:
match_count = (
    joined.groupby("관리번호")
    .size()
    .value_counts()
    .sort_index()
)

print(match_count)

In [ ]:
print(commercial_gdf["TRDAR_SE_1"].value_counts())

print(
    joined["TRDAR_SE_1"].value_counts(dropna=False)
)

print(
    duplicates[
        ["관리번호", "사업장명", "TRDAR_SE_1", "TRDAR_CD", "TRDAR_CD_N"]
    ]
    .sort_values("관리번호")
    .head(30)
)

In [ ]:
dup_types = (
    duplicates
    .groupby("관리번호")["TRDAR_SE_1"]
    .apply(lambda x: " + ".join(sorted(x.astype(str))))
    .value_counts()
)

print(dup_types)

In [ ]:
tourism_ids = set(
    joined.loc[
        joined["TRDAR_SE_1"] == "관광특구",
        "관리번호"
    ]
)

joined["관광특구여부"] = (
    joined["관리번호"].isin(tourism_ids)
).astype(int)

print(joined["관광특구여부"].value_counts())

In [ ]:
priority = {
    "골목상권": 1,
    "발달상권": 2,
    "전통시장": 3,
    "관광특구": 4
}

joined["상권우선순위"] = joined["TRDAR_SE_1"].map(priority)

In [ ]:
joined_clean = (
    joined
    .sort_values(
        ["관리번호", "상권우선순위"],
        na_position="last"
    )
    .drop_duplicates(
        subset="관리번호",
        keep="first"
    )
    .copy()
)

In [ ]:
print("정리 전:", len(joined))
print("정리 후:", len(joined_clean))

print(
    "관리번호 중복:",
    joined_clean["관리번호"].duplicated().sum()
)

In [ ]:
print(
    joined_clean["TRDAR_SE_1"].value_counts(dropna=False)
)

print()

print(
    "최종 상권 매칭 성공:",
    joined_clean["TRDAR_CD"].notna().sum()
)

print(
    "최종 상권 매칭 실패:",
    joined_clean["TRDAR_CD"].isna().sum()
)

print(
    "최종 매칭률:",
    round(
        joined_clean["TRDAR_CD"].notna().mean() * 100,
        2
    ),
    "%"
)

In [ ]:
print(
    joined_clean["관광특구여부"].value_counts()
)

## 3. Store-Level Commercial Area Features

2021~2025년 점포-상권 파일을 통합한 뒤 음식점 업태와 매핑하고, 개업 시점의 분기·상권·업태 기준으로 점포 특성을 결합합니다.

In [ ]:
import glob
import os

csv_files = glob.glob(str(DATA_DIR / "*.csv"))

for f in csv_files:
    print(os.path.basename(f))

In [ ]:
import unicodedata

store_files = []

for f in csv_files:
    name = unicodedata.normalize(
        "NFC",
        os.path.basename(f)
    )

    if (
        "점포-상권" in name
        and any(f"{year}년" in name for year in range(2021, 2026))
    ):
        store_files.append(f)

store_files = sorted(store_files)

print("찾은 파일 수:", len(store_files))

for f in store_files:
    print(os.path.basename(f))

In [ ]:
import pandas as pd
import os

# 영문 컬럼 → 한글 컬럼 통일
rename_map = {
    "stdr_yyqu_cd": "기준_년분기_코드",
    "trdar_se_cd": "상권_구분_코드",
    "trdar_se_cd_nm": "상권_구분_코드_명",
    "trdar_cd": "상권_코드",
    "trdar_cd_nm": "상권_코드_명",
    "svc_induty_cd": "서비스_업종_코드",
    "svc_induty_cd_nm": "서비스_업종_코드_명",
    "stor_co": "점포_수",
    "similr_induty_stor_co": "유사_업종_점포_수",
    "opbiz_rt": "개업_율",
    "opbiz_stor_co": "개업_점포_수",
    "clsbiz_rt": "폐업_률",
    "clsbiz_stor_co": "폐업_점포_수",
    "frc_stor_co": "프랜차이즈_점포_수"
}

store_list = []

for file in store_files:

    temp = pd.read_csv(
        file,
        encoding="euc-kr",
        encoding_errors="replace",
        low_memory=False
    )

    # 영문 컬럼이 있으면 한글로 변경
    temp = temp.rename(columns=rename_map)

    store_list.append(temp)

    print(
        os.path.basename(file),
        temp.shape
    )

In [ ]:
store_all = pd.concat(
    store_list,
    ignore_index=True
)

print("합친 데이터 크기:", store_all.shape)

for i, col in enumerate(store_all.columns):
    print(i, col)

In [ ]:
print(
    sorted(
        store_all["기준_년분기_코드"]
        .dropna()
        .unique()
    )
)

In [ ]:
store_type_map = {
    "한식음식점": "한식",
    "양식음식점": "경양식",
    "일식음식점": "일식",
    "중식음식점": "중국식",
    "분식전문점": "분식",
    "치킨전문점": "호프/통닭",
    "호프-간이주점": "호프/통닭"
}

store_all["분석업태"] = (
    store_all["서비스_업종_코드_명"]
    .map(store_type_map)
)

store_food_all = store_all[
    store_all["분석업태"].notna()
].copy()

print(store_food_all["분석업태"].value_counts())

In [ ]:
store_food_all["상권_코드"] = pd.to_numeric(
    store_food_all["상권_코드"],
    errors="coerce"
).astype("Int64")

store_food_all["기준_년분기_코드"] = pd.to_numeric(
    store_food_all["기준_년분기_코드"],
    errors="coerce"
).astype("Int64")

In [ ]:
store_grouped_all = (
    store_food_all
    .groupby(
        [
            "기준_년분기_코드",
            "상권_코드",
            "분석업태"
        ],
        as_index=False
    )
    .agg({
        "점포_수": "sum",
        "유사_업종_점포_수": "sum",
        "개업_점포_수": "sum",
        "폐업_점포_수": "sum",
        "프랜차이즈_점포_수": "sum"
    })
)

print(store_grouped_all.head(20))
print(store_grouped_all.shape)

In [ ]:
store_grouped_all["프랜차이즈_비율"] = (
    store_grouped_all["프랜차이즈_점포_수"]
    / store_grouped_all["점포_수"]
)

store_grouped_all["개업률_계산"] = (
    store_grouped_all["개업_점포_수"]
    / store_grouped_all["점포_수"]
)

store_grouped_all["폐업률_계산"] = (
    store_grouped_all["폐업_점포_수"]
    / store_grouped_all["점포_수"]
)

In [ ]:
analysis_df = joined_clean.merge(
    store_grouped_all,
    how="left",
    left_on=[
        "기준_년분기_코드",
        "TRDAR_CD",
        "업태구분명"
    ],
    right_on=[
        "기준_년분기_코드",
        "상권_코드",
        "분석업태"
    ]
)

print("merge 전:", len(joined_clean))
print("merge 후:", len(analysis_df))

In [ ]:
print(
    "점포정보 매칭 성공:",
    analysis_df["점포_수"].notna().sum()
)

print(
    "점포정보 매칭 실패:",
    analysis_df["점포_수"].isna().sum()
)

print(
    "전체 점포정보 매칭률:",
    round(
        analysis_df["점포_수"].notna().mean() * 100,
        2
    ),
    "%"
)

In [ ]:
print(
    "상권 매칭 음식점 중 점포정보 매칭률:",
    round(
        analysis_df.loc[
            analysis_df["TRDAR_CD"].notna(),
            "점포_수"
        ].notna().mean() * 100,
        2
    ),
    "%"
)

In [ ]:
year_match = (
    analysis_df
    .groupby("개업연도")
    .agg(
        음식점수=("관리번호", "size"),
        상권매칭수=("TRDAR_CD", lambda x: x.notna().sum()),
        점포정보매칭수=("점포_수", lambda x: x.notna().sum())
    )
)

year_match["전체_점포매칭률"] = (
    year_match["점포정보매칭수"]
    / year_match["음식점수"]
    * 100
).round(2)

year_match["상권내_점포매칭률"] = (
    year_match["점포정보매칭수"]
    / year_match["상권매칭수"]
    * 100
).round(2)

print(year_match)

In [ ]:
print(
    analysis_df[
        [
            "사업장명",
            "인허가일자",
            "업태구분명",
            "TRDAR_CD_N",
            "기준_년분기_코드",
            "점포_수",
            "유사_업종_점포_수",
            "개업_점포_수",
            "폐업_점포_수",
            "프랜차이즈_점포_수"
        ]
    ].head(30)
)

### 3.1 Derived store features

프랜차이즈 비율, 개업점포 비율, 폐업점포 비율을 생성합니다.

In [ ]:
analysis_df["프랜차이즈_비율"] = (
    analysis_df["프랜차이즈_점포_수"]
    / analysis_df["점포_수"]
)

analysis_df["개업점포_비율"] = (
    analysis_df["개업_점포_수"]
    / analysis_df["점포_수"]
)

analysis_df["폐업점포_비율"] = (
    analysis_df["폐업_점포_수"]
    / analysis_df["점포_수"]
)

In [ ]:
import numpy as np

for col in [
    "프랜차이즈_비율",
    "개업점포_비율",
    "폐업점포_비율"
]:
    analysis_df[col] = analysis_df[col].replace(
        [np.inf, -np.inf],
        np.nan
    )

## 4. Estimated Sales Features

2021~2025년 추정매출-상권 파일을 통합하고 상권·분기·업태 기준 매출을 결합한 뒤 점포당 매출 지표를 생성합니다.

In [ ]:
import glob
import os
import unicodedata

csv_files = glob.glob(str(DATA_DIR / "*.csv"))

sales_files = []

for f in csv_files:
    name = unicodedata.normalize(
        "NFC",
        os.path.basename(f)
    )

    if (
        "추정매출-상권" in name
        and any(f"{year}년" in name for year in range(2021, 2026))
    ):
        sales_files.append(f)

sales_files = sorted(sales_files)

print("찾은 파일 수:", len(sales_files))

for f in sales_files:
    print(os.path.basename(f))

In [ ]:
sales_list = []

for file in sales_files:
    temp = pd.read_csv(
        file,
        encoding="euc-kr",
        encoding_errors="replace",
        low_memory=False
    )

    sales_list.append(temp)

    print(
        os.path.basename(file),
        temp.shape
    )

In [ ]:
sales_all = pd.concat(
    sales_list,
    ignore_index=True
)

print("합친 데이터 크기:", sales_all.shape)

In [ ]:
periods = sorted(
    sales_all["기준_년분기_코드"]
    .dropna()
    .unique()
)

print(periods)

In [ ]:
for i, col in enumerate(sales_all.columns):
    print(i, col)

In [ ]:
sales_type_map = {
    "한식음식점": "한식",
    "양식음식점": "경양식",
    "일식음식점": "일식",
    "중식음식점": "중국식",
    "분식전문점": "분식",
    "치킨전문점": "호프/통닭",
    "호프-간이주점": "호프/통닭"
}

sales_all["분석업태"] = (
    sales_all["서비스_업종_코드_명"]
    .map(sales_type_map)
)

print(sales_all["분석업태"].value_counts(dropna=False))

In [ ]:
sales_food = sales_all[
    sales_all["분석업태"].notna()
].copy()

print(sales_food["분석업태"].value_counts())
print("음식 관련 행 수:", len(sales_food))

In [ ]:
sales_food["상권_코드"] = pd.to_numeric(
    sales_food["상권_코드"],
    errors="coerce"
).astype("Int64")

sales_food["기준_년분기_코드"] = pd.to_numeric(
    sales_food["기준_년분기_코드"],
    errors="coerce"
).astype("Int64")

analysis_df["TRDAR_CD"] = pd.to_numeric(
    analysis_df["TRDAR_CD"],
    errors="coerce"
).astype("Int64")

analysis_df["기준_년분기_코드"] = pd.to_numeric(
    analysis_df["기준_년분기_코드"],
    errors="coerce"
).astype("Int64")

In [ ]:
sales_food["당월_매출_금액"] = pd.to_numeric(
    sales_food["당월_매출_금액"],
    errors="coerce"
)

sales_food["당월_매출_건수"] = pd.to_numeric(
    sales_food["당월_매출_건수"],
    errors="coerce"
)

print(
    sales_food[
        ["당월_매출_금액", "당월_매출_건수"]
    ].describe()
)

In [ ]:
sales_grouped = (
    sales_food
    .groupby(
        [
            "기준_년분기_코드",
            "상권_코드",
            "분석업태"
        ],
        as_index=False
    )
    .agg({
        "당월_매출_금액": "sum",
        "당월_매출_건수": "sum"
    })
)

print(sales_grouped.head())
print(sales_grouped.shape)

In [ ]:
print(
    "중복 key:",
    sales_grouped.duplicated(
        subset=[
            "기준_년분기_코드",
            "상권_코드",
            "분석업태"
        ]
    ).sum()
)

In [ ]:
analysis_df = analysis_df.merge(
    sales_grouped,
    how="left",
    left_on=[
        "기준_년분기_코드",
        "TRDAR_CD",
        "업태구분명"
    ],
    right_on=[
        "기준_년분기_코드",
        "상권_코드",
        "분석업태"
    ],
    suffixes=("", "_sales")
)

print("행 수:", len(analysis_df))

In [ ]:
analysis_df["점포당_매출금액"] = (
    analysis_df["당월_매출_금액"]
    / analysis_df["점포_수"]
)

analysis_df["점포당_매출건수"] = (
    analysis_df["당월_매출_건수"]
    / analysis_df["점포_수"]
)

In [ ]:
print(
    "매출정보 매칭률:",
    round(
        analysis_df["당월_매출_금액"].notna().mean() * 100,
        2
    ),
    "%"
)

print(
    "상권 내 매출정보 매칭률:",
    round(
        analysis_df.loc[
            analysis_df["TRDAR_CD"].notna(),
            "당월_매출_금액"
        ].notna().mean() * 100,
        2
    ),
    "%"
)

In [ ]:
sales_year_match = (
    analysis_df
    .groupby("개업연도")
    .agg(
        음식점수=("관리번호", "size"),
        상권매칭수=("TRDAR_CD", lambda x: x.notna().sum()),
        매출매칭수=("당월_매출_금액", lambda x: x.notna().sum())
    )
)

sales_year_match["전체_매출매칭률"] = (
    sales_year_match["매출매칭수"]
    / sales_year_match["음식점수"]
    * 100
).round(2)

sales_year_match["상권내_매출매칭률"] = (
    sales_year_match["매출매칭수"]
    / sales_year_match["상권매칭수"]
    * 100
).round(2)

print(sales_year_match)

In [ ]:
print(
    analysis_df[
        [
            "사업장명",
            "업태구분명",
            "TRDAR_CD_N",
            "점포_수",
            "당월_매출_금액",
            "당월_매출_건수",
            "점포당_매출금액",
            "점포당_매출건수"
        ]
    ].head(20)
)

## 5. Floating Population Features

길단위인구-상권 데이터에서 20·30대 비율, 점심시간 비율, 저녁·야간 비율을 생성합니다.

In [ ]:
pop_path = DATA_DIR / "서울시 상권분석서비스(길단위인구-상권).csv"

pop = pd.read_csv(
    pop_path,
    encoding="euc-kr",
    encoding_errors="replace",
    low_memory=False
)

print("유동인구 데이터 크기:", pop.shape)


In [ ]:
pop_small = pop[
    [
        "기준_년분기_코드",
        "상권_코드",
        "총_유동인구_수",
        "연령대_20_유동인구_수",
        "연령대_30_유동인구_수",
        "시간대_11_14_유동인구_수",
        "시간대_17_21_유동인구_수",
        "시간대_21_24_유동인구_수"
    ]
].copy()

print(pop_small.head())
print(pop_small.shape)

In [ ]:
pop_small["상권_코드"] = pd.to_numeric(
    pop_small["상권_코드"],
    errors="coerce"
).astype("Int64")

pop_small["기준_년분기_코드"] = pd.to_numeric(
    pop_small["기준_년분기_코드"],
    errors="coerce"
).astype("Int64")

In [ ]:
pop_cols = [
    "총_유동인구_수",
    "연령대_20_유동인구_수",
    "연령대_30_유동인구_수",
    "시간대_11_14_유동인구_수",
    "시간대_17_21_유동인구_수",
    "시간대_21_24_유동인구_수"
]

for col in pop_cols:
    pop_small[col] = pd.to_numeric(
        pop_small[col],
        errors="coerce"
    )

In [ ]:
print(
    "중복 key:",
    pop_small.duplicated(
        subset=[
            "기준_년분기_코드",
            "상권_코드"
        ]
    ).sum()
)

In [ ]:
analysis_df = analysis_df.merge(
    pop_small,
    how="left",
    left_on=[
        "기준_년분기_코드",
        "TRDAR_CD"
    ],
    right_on=[
        "기준_년분기_코드",
        "상권_코드"
    ],
    suffixes=("", "_pop")
)

print("행 수:", len(analysis_df))

In [ ]:
print(
    "유동인구 매칭률:",
    round(
        analysis_df["총_유동인구_수"]
        .notna()
        .mean() * 100,
        2
    ),
    "%"
)

print(
    "상권 내 유동인구 매칭률:",
    round(
        analysis_df.loc[
            analysis_df["TRDAR_CD"].notna(),
            "총_유동인구_수"
        ].notna().mean() * 100,
        2
    ),
    "%"
)

In [ ]:
analysis_df["20_30대_유동인구_비율"] = (
    analysis_df["연령대_20_유동인구_수"]
    + analysis_df["연령대_30_유동인구_수"]
) / analysis_df["총_유동인구_수"]

analysis_df["점심시간_유동인구_비율"] = (
    analysis_df["시간대_11_14_유동인구_수"]
    / analysis_df["총_유동인구_수"]
)

analysis_df["저녁야간_유동인구_비율"] = (
    analysis_df["시간대_17_21_유동인구_수"]
    + analysis_df["시간대_21_24_유동인구_수"]
) / analysis_df["총_유동인구_수"]

## 6. Resident Population Features

상주인구-상권 데이터에서 20·30대 상주인구 비율과 가구당 상주인구를 생성합니다.

In [ ]:
resident_path = DATA_DIR / "서울시 상권분석서비스(상주인구-상권).csv"

resident = pd.read_csv(
    resident_path,
    encoding="euc-kr",
    encoding_errors="replace",
    low_memory=False
)

print(resident.shape)

for i, col in enumerate(resident.columns):
    print(i, col)

In [ ]:
print(
    sorted(
        resident["기준_년분기_코드"]
        .dropna()
        .unique()
    )
)

In [ ]:
for col in resident.columns:
    if "인구" in col or "가구" in col:
        print(col)

In [ ]:
resident_small = resident[
    [
        "기준_년분기_코드",
        "상권_코드",
        "총_상주인구_수",
        "연령대_20_상주인구_수",
        "연령대_30_상주인구_수",
        "총_가구_수"
    ]
].copy()

print(resident_small.head())
print(resident_small.shape)

In [ ]:
resident_small["상권_코드"] = pd.to_numeric(
    resident_small["상권_코드"],
    errors="coerce"
).astype("Int64")

resident_small["기준_년분기_코드"] = pd.to_numeric(
    resident_small["기준_년분기_코드"],
    errors="coerce"
).astype("Int64")

In [ ]:
resident_cols = [
    "총_상주인구_수",
    "연령대_20_상주인구_수",
    "연령대_30_상주인구_수",
    "총_가구_수"
]

for col in resident_cols:
    resident_small[col] = pd.to_numeric(
        resident_small[col],
        errors="coerce"
    )

In [ ]:
print(
    "중복 key:",
    resident_small.duplicated(
        subset=[
            "기준_년분기_코드",
            "상권_코드"
        ]
    ).sum()
)

In [ ]:
analysis_df = analysis_df.merge(
    resident_small,
    how="left",
    left_on=[
        "기준_년분기_코드",
        "TRDAR_CD"
    ],
    right_on=[
        "기준_년분기_코드",
        "상권_코드"
    ],
    suffixes=("", "_resident")
)

print("행 수:", len(analysis_df))

In [ ]:
print(
    "상주인구 매칭률:",
    round(
        analysis_df["총_상주인구_수"]
        .notna()
        .mean() * 100,
        2
    ),
    "%"
)

print(
    "상권 내 상주인구 매칭률:",
    round(
        analysis_df.loc[
            analysis_df["TRDAR_CD"].notna(),
            "총_상주인구_수"
        ].notna().mean() * 100,
        2
    ),
    "%"
)

In [ ]:
analysis_df["20_30대_상주인구_비율"] = (
    analysis_df["연령대_20_상주인구_수"]
    + analysis_df["연령대_30_상주인구_수"]
) / analysis_df["총_상주인구_수"]

analysis_df["가구당_상주인구"] = (
    analysis_df["총_상주인구_수"]
    / analysis_df["총_가구_수"]
)

## 7. Modeling Dataset

결측 현황을 확인하고 상권 매칭 및 핵심 변수 완전자료를 기준으로 생존분석용 데이터셋을 구성합니다.

In [ ]:
print("전체 행 수:", len(analysis_df))
print("전체 열 수:", analysis_df.shape[1])

print("\n[event]")
print(analysis_df["event"].value_counts())

print("\n[duration]")
print(analysis_df["duration_days"].describe())

In [ ]:
check_cols = [
    "duration_days",
    "event",
    "업태구분명",
    "소재지면적",

    "점포_수",
    "개업_점포_수",
    "폐업_점포_수",
    "프랜차이즈_점포_수",

    "당월_매출_금액",
    "당월_매출_건수",
    "점포당_매출금액",

    "총_유동인구_수",
    "20_30대_유동인구_비율",
    "점심시간_유동인구_비율",
    "저녁야간_유동인구_비율",

    "총_상주인구_수",
    "20_30대_상주인구_비율",
    "총_가구_수"
]

missing_table = pd.DataFrame({
    "결측수": analysis_df[check_cols].isna().sum(),
    "결측률(%)": (
        analysis_df[check_cols].isna().mean() * 100
    ).round(2)
})

print(missing_table)

In [ ]:
model_df = analysis_df[
    analysis_df["TRDAR_CD"].notna()
].copy()

print("전체 음식점:", len(analysis_df))
print("상권 매칭 음식점:", len(model_df))

In [ ]:
model_cols = [
    "duration_days",
    "event",
    "업태구분명",

    "점포_수",
    "폐업_점포_수",
    "프랜차이즈_점포_수",

    "점포당_매출금액",

    "총_유동인구_수",
    "20_30대_유동인구_비율",

    "총_상주인구_수"
]

complete_df = model_df.dropna(
    subset=model_cols
).copy()

print("상권 매칭 데이터:", len(model_df))
print("핵심 변수 완전자료:", len(complete_df))

print(
    "사용 가능 비율:",
    round(len(complete_df) / len(model_df) * 100, 2),
    "%"
)

## 8. Kaplan–Meier Survival Analysis

전체 음식점과 업태별 생존곡선을 추정하고, 업태 간 차이는 log-rank test로 검정합니다.

In [ ]:
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt

kmf = KaplanMeierFitter()

kmf.fit(
    durations=complete_df["duration_days"],
    event_observed=complete_df["event"],
    label="전체 음식점"
)

kmf.plot_survival_function()

plt.xlabel("개업 후 경과일")
plt.ylabel("생존확률")
plt.title("seoul restaurant Kaplan-Meier survival curve")
plt.show()

In [ ]:
for day in [180, 365, 730, 1095, 1460]:
    survival_prob = kmf.predict(day)

    print(
        f"{day}일 생존확률:",
        round(survival_prob, 4)
    )

In [ ]:
industry_name_map = {
    "한식": "Korean",
    "경양식": "Western",
    "일식": "Japanese",
    "중국식": "Chinese",
    "분식": "Snack Food",
    "호프/통닭": "Pub/Chicken"
}

In [ ]:
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 7))

for industry in complete_df["업태구분명"].unique():

    temp = complete_df[
        complete_df["업태구분명"] == industry
    ]

    kmf = KaplanMeierFitter()

    kmf.fit(
        durations=temp["duration_days"],
        event_observed=temp["event"],
        label=industry_name_map[industry]
    )

    kmf.plot_survival_function(
        ci_show=False
    )

plt.xlabel("Days Since Opening")
plt.ylabel("Survival Probability")
plt.title("Kaplan-Meier Survival Curves by Restaurant Type")
plt.legend(title="Restaurant Type")

plt.show()

In [ ]:
results = []

for industry in complete_df["업태구분명"].unique():

    temp = complete_df[
        complete_df["업태구분명"] == industry
    ]

    kmf = KaplanMeierFitter()

    kmf.fit(
        durations=temp["duration_days"],
        event_observed=temp["event"]
    )

    results.append({
        "Restaurant Type": industry_name_map[industry],
        "Sample Size": len(temp),
        "1-Year Survival": kmf.predict(365),
        "2-Year Survival": kmf.predict(730),
        "3-Year Survival": kmf.predict(1095)
    })

industry_survival = pd.DataFrame(results)

industry_survival = industry_survival.sort_values(
    "3-Year Survival",
    ascending=False
)

print(industry_survival)

In [ ]:
industry_survival[
    [
        "1-Year Survival",
        "2-Year Survival",
        "3-Year Survival"
    ]
] = industry_survival[
    [
        "1-Year Survival",
        "2-Year Survival",
        "3-Year Survival"
    ]
].round(3)

print(industry_survival)

In [ ]:
from lifelines.statistics import multivariate_logrank_test

result = multivariate_logrank_test(
    complete_df["duration_days"],
    complete_df["업태구분명"],
    complete_df["event"]
)

print(result)

## 9. Cox Proportional Hazards Model

연속형 규모 변수는 `log1p` 변환하고, 업태를 포함한 초기 Cox PH 모델을 적합한 뒤 비례위험 가정을 점검합니다.

In [ ]:
cox_cols = [
    "duration_days",
    "event",
    "업태구분명",

    "점포_수",
    "폐업점포_비율",
    "프랜차이즈_비율",

    "점포당_매출금액",

    "총_유동인구_수",
    "20_30대_유동인구_비율",
    "점심시간_유동인구_비율",
    "저녁야간_유동인구_비율",

    "총_상주인구_수",
    "20_30대_상주인구_비율",

    "관광특구여부"
]

cox_df = analysis_df[cox_cols].dropna().copy()

print("Cox sample size:", len(cox_df))
print(cox_df["event"].value_counts())

In [ ]:
import numpy as np

cox_df["log_store_count"] = np.log1p(
    cox_df["점포_수"]
)

cox_df["log_sales_per_store"] = np.log1p(
    cox_df["점포당_매출금액"]
)

cox_df["log_floating_population"] = np.log1p(
    cox_df["총_유동인구_수"]
)

cox_df["log_resident_population"] = np.log1p(
    cox_df["총_상주인구_수"]
)

In [ ]:
cox_df = cox_df.drop(
    columns=[
        "점포_수",
        "점포당_매출금액",
        "총_유동인구_수",
        "총_상주인구_수"
    ]
)

In [ ]:
cox_df = pd.get_dummies(
    cox_df,
    columns=["업태구분명"],
    drop_first=True,
    dtype=int
)

print(cox_df.columns.tolist())
print(cox_df.shape)

In [ ]:
from lifelines import CoxPHFitter

cph = CoxPHFitter()

cph.fit(
    cox_df,
    duration_col="duration_days",
    event_col="event"
)

cph.print_summary()

In [ ]:
cph.check_assumptions(
    cox_df,
    p_value_threshold=0.05,
    show_plots=False
)

In [ ]:
hr = 1.46

hr_10pp = hr ** 0.1

print(hr_10pp)
print("위험 변화율:", (hr_10pp - 1) * 100, "%")

### 9.1 Stratified Cox model

비례위험 가정을 보완하기 위해 업태를 층화변수로 사용합니다.

In [ ]:
cox_cols2 = [
    "duration_days",
    "event",
    "업태구분명",

    "폐업점포_비율",
    "프랜차이즈_비율",
    "20_30대_유동인구_비율",
    "점심시간_유동인구_비율",
    "저녁야간_유동인구_비율",
    "20_30대_상주인구_비율",
    "관광특구여부",

    "점포_수",
    "점포당_매출금액",
    "총_유동인구_수",
    "총_상주인구_수"
]

cox_df2 = analysis_df[cox_cols2].dropna().copy()

print(cox_df2.shape)

In [ ]:
import numpy as np

cox_df2["log_store_count"] = np.log1p(
    cox_df2["점포_수"]
)

cox_df2["log_sales_per_store"] = np.log1p(
    cox_df2["점포당_매출금액"]
)

cox_df2["log_floating_population"] = np.log1p(
    cox_df2["총_유동인구_수"]
)

cox_df2["log_resident_population"] = np.log1p(
    cox_df2["총_상주인구_수"]
)

cox_df2 = cox_df2.drop(
    columns=[
        "점포_수",
        "점포당_매출금액",
        "총_유동인구_수",
        "총_상주인구_수"
    ]
)

In [ ]:
from lifelines import CoxPHFitter

cph2 = CoxPHFitter()

cph2.fit(
    cox_df2,
    duration_col="duration_days",
    event_col="event",
    strata=["업태구분명"]
)

cph2.print_summary()

In [ ]:
cph2.check_assumptions(
    cox_df2,
    p_value_threshold=0.05,
    show_plots=False
)

### 9.2 Lunch-time population strata

점심시간 유동인구 비율을 4분위로 구간화하여 층화합니다.

In [ ]:
# 점심시간 유동인구 비율을 4분위 구간으로 나누기
cox_df2["lunch_ratio_group"] = pd.qcut(
    cox_df2["점심시간_유동인구_비율"],
    q=4,
    labels=["Q1", "Q2", "Q3", "Q4"]
)

print(cox_df2["lunch_ratio_group"].value_counts())

In [ ]:
cox_df3 = cox_df2.drop(
    columns=["점심시간_유동인구_비율"]
).copy()

In [ ]:
from lifelines import CoxPHFitter

cph3 = CoxPHFitter()

cph3.fit(
    cox_df3,
    duration_col="duration_days",
    event_col="event",
    strata=[
        "업태구분명",
        "관광특구여부",
        "lunch_ratio_group"
    ]
)

cph3.print_summary()

In [ ]:
cph3.check_assumptions(
    cox_df3,
    p_value_threshold=0.05,
    show_plots=False
)

### 9.3 Final Cox model

가정 점검 결과를 반영하여 최종 설명변수 집합으로 Cox 모델을 적합합니다.

In [ ]:
cox_df4 = cox_df3.drop(
    columns=[
        "log_store_count",
        "저녁야간_유동인구_비율"
    ]
).copy()

print(cox_df4.columns)
print(cox_df4.shape)

In [ ]:
from lifelines import CoxPHFitter

cph_final = CoxPHFitter()

cph_final.fit(
    cox_df4,
    duration_col="duration_days",
    event_col="event",
    strata=[
        "업태구분명",
        "관광특구여부",
        "lunch_ratio_group"
    ]
)

cph_final.print_summary()

In [ ]:
cph_final.check_assumptions(
    cox_df4,
    p_value_threshold=0.05,
    show_plots=False
)

In [ ]:
hr = 1.86

hr_10pp = hr ** 0.1

print("HR per 10 percentage points:", hr_10pp)
print("Change:", (hr_10pp - 1) * 100, "%")

## 10. Hazard Ratio Forest Plot

최종 Cox 모델의 Hazard Ratio와 95% 신뢰구간을 시각화합니다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Cox 결과 가져오기
forest_df = cph_final.summary[
    [
        "exp(coef)",
        "exp(coef) lower 95%",
        "exp(coef) upper 95%",
        "p"
    ]
].copy()

forest_df.columns = [
    "HR",
    "CI_lower",
    "CI_upper",
    "p_value"
]

# 보기 좋은 변수명으로 변경
name_map = {
    "폐업점포_비율": "Closure Rate",
    "프랜차이즈_비율": "Franchise Ratio",
    "20_30대_유동인구_비율": "Floating Pop. Age 20-30 Ratio",
    "20_30대_상주인구_비율": "Resident Pop. Age 20-30 Ratio",
    "log_sales_per_store": "Sales per Store (log)",
    "log_floating_population": "Floating Population (log)",
    "log_resident_population": "Resident Population (log)"
}

forest_df["Variable"] = [
    name_map.get(idx, idx)
    for idx in forest_df.index
]

# 그래프 순서 뒤집기
forest_df = forest_df.iloc[::-1].reset_index(drop=True)

forest_df

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

y_pos = np.arange(len(forest_df))

# Hazard Ratio 점
ax.scatter(
    forest_df["HR"],
    y_pos,
    s=60,
    zorder=3
)

# 95% CI
ax.errorbar(
    forest_df["HR"],
    y_pos,
    xerr=[
        forest_df["HR"] - forest_df["CI_lower"],
        forest_df["CI_upper"] - forest_df["HR"]
    ],
    fmt="none",
    capsize=4,
    zorder=2
)

# HR = 1 기준선
ax.axvline(
    x=1,
    linestyle="--",
    linewidth=1
)

ax.set_yticks(y_pos)
ax.set_yticklabels(forest_df["Variable"])

ax.set_xlabel("Hazard Ratio (95% CI)")
ax.set_title("Hazard Ratio Forest Plot")

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

y_pos = np.arange(len(forest_df))

ax.scatter(
    forest_df["HR"],
    y_pos,
    s=60,
    zorder=3
)

ax.errorbar(
    forest_df["HR"],
    y_pos,
    xerr=[
        forest_df["HR"] - forest_df["CI_lower"],
        forest_df["CI_upper"] - forest_df["HR"]
    ],
    fmt="none",
    capsize=4,
    zorder=2
)

ax.axvline(
    x=1,
    linestyle="--",
    linewidth=1
)

ax.set_yticks(y_pos)
ax.set_yticklabels(forest_df["Variable"])

ax.set_xlabel("Hazard Ratio (95% CI)")
ax.set_title("Cox Proportional Hazards Model")

# 오른쪽에 HR, CI, p-value 표시
max_ci = forest_df["CI_upper"].max()

for i, row in forest_df.iterrows():

    if row["p_value"] < 0.001:
        p_text = "p < 0.001"
    else:
        p_text = f"p = {row['p_value']:.3f}"

    text = (
        f"HR {row['HR']:.2f} "
        f"({row['CI_lower']:.2f}-{row['CI_upper']:.2f}), "
        f"{p_text}"
    )

    ax.text(
        max_ci + 0.08,
        i,
        text,
        va="center",
        fontsize=9
    )

ax.set_xlim(
    min(0.8, forest_df["CI_lower"].min() - 0.05),
    max_ci + 1.1
)

plt.tight_layout()
plt.show()